In [1]:
import dspy
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
import ollama
from typing import List, Union
import numpy as np

# Konfiguration des lokalen LLMs (z.B. Qwen oder Llama 3 via Ollama)
local_llm = dspy.LM(
    "openai/qwen3:30b",
    api_base="http://localhost:11434/v1", 
    api_key="no_key_needed"
)

dspy.configure(lm=local_llm)

In [2]:
# Beispieldaten (Wissensbasis)
documents = [
    "Titus Skates wurde in den späten 1970er Jahren von dem deutschen Skateboard-Pionier Titus Dittmann gegründet.",
    "Der Münsteraner Unternehmer Titus Dittmann organisierte den legendären Münster Monster Mastership.",
    "Skateboards bestehen typischerweise aus sieben Lagen kanadischem Ahornholz, Rollen aus Polyurethan und Achsen aus Aluminium.",
    "Apple wurde 1976 von Steve Jobs, Steve Wozniak und Ronald Wayne gegründet.",
    "Die Stadt Münster ist bekannt für ihre historische Altstadt und das Picasso-Museum.",
    "Der Mond umkreist die Erde in etwa 27,3 Tagen."
]

# Qdrant In-Memory Client initialisieren
qdrant_client = QdrantClient(":memory:")
collection_name = "my_knowledge_base"

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

# Vektorisierung und Upload der Dokumente
points = []
for idx, doc in enumerate(documents):
    # Embedding-Modell muss lokal in Ollama verfügbar sein (z.B. 'nomic-embed-text' oder 'embeddinggemma')
    vector = ollama.embed(model="embeddinggemma", input=doc).embeddings[0]
    points.append(PointStruct(id=idx, vector=vector, payload={"text": doc}))

qdrant_client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [3]:
class QdrantRM(dspy.Retrieve):
    def __init__(self, client, collection_name, k=3):
        super().__init__(k=k)
        self.client = client
        self.collection_name = collection_name

    def forward(self, query_or_queries: Union[str, List[str]], k=None) -> dspy.Prediction:
        k = k if k is not None else self.k
        query = query_or_queries if isinstance(query_or_queries, str) else query_or_queries[0]

        # Korrektur: Nutzung der 'query' Variable für das Embedding
        query_vector = ollama.embed(model="embeddinggemma", input=query).embeddings[0]

        search_result = self.client.query_points(
            collection_name=self.collection_name,
            query=np.array(query_vector),
            limit=k
        )

        passages = [hit.payload['text'] for hit in search_result.points]
        return dspy.Prediction(passages=passages)

# Initialisierung des Retrievers
my_retriever = QdrantRM(client=qdrant_client, collection_name=collection_name, k=2)

In [4]:
# Signatur für die Erzeugung der hypothetischen Antwort
class GenerateHypothetical(dspy.Signature):
    """Schreibe eine hypothetische Antwort auf die Frage. Sie muss nicht faktisch korrekt sein, 
    sondern Keywords und Struktur enthalten, die in einem relevanten Dokument vorkommen könnten."""

    question = dspy.InputField(desc="Die Frage des Nutzers")
    hypothetical_answer = dspy.OutputField(desc="Eine plausible, hypothetische Antwort")

# Signatur für die finale Antwort (wie an Tag 20)
class GenerateAnswer(dspy.Signature):
    """Beantworte Fragen präzise basierend auf dem gegebenen Kontext."""

    context = dspy.InputField(desc="Fakten aus der Wissensdatenbank")
    question = dspy.InputField(desc="Die Frage des Nutzers")
    answer = dspy.OutputField(desc="Die präzise Antwort")

# Das erweiterte RAG-Modul
class RAGWithHyDE(dspy.Module):
    def __init__(self, retriever_model):
        super().__init__()
        # Schritt 1: HyDE Generator
        self.generate_hyde = dspy.ChainOfThought(GenerateHypothetical)

        # Schritt 2: Retriever
        self.retrieve = retriever_model

        # Schritt 3: Finale Antwort
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        # 1. Generiere hypothetisches Dokument
        hyde_output = self.generate_hyde(question=question)
        hypothetical_doc = hyde_output.hypothetical_answer

        # 2. Nutze das hypothetische Dokument für die Suche (statt der Frage)
        # Dies hilft, Dokumente zu finden, die der Antwortstruktur ähneln
        retrieval_result = self.retrieve(hypothetical_doc)
        context = retrieval_result.passages

        # 3. Generiere die finale Antwort mit dem gefundenen Kontext und der Originalfrage
        prediction = self.generate_answer(context=context, question=question)

        return dspy.Prediction(
            context=context, 
            answer=prediction.answer, 
            hypothetical_used=hypothetical_doc
        )

In [5]:
# System initialisieren
rag_hyde_system = RAGWithHyDE(retriever_model=my_retriever)

# Frage, die Kontextwissen erfordert
question = "Was hat der Gründer von Titus Skates organisiert?"

# Ausführung
response = rag_hyde_system(question)

print(f"--- ERGEBNIS ---\n")
print(f"Frage: {question}")
print(f"Hypothetische Antwort (für Suche genutzt): {response.hypothetical_used}\n")
print(f"Gefundener Kontext: {response.context}\n")
print(f"Finale Antwort: {response.answer}\n")

--- ERGEBNIS ---

Frage: Was hat der Gründer von Titus Skates organisiert?
Hypothetische Antwort (für Suche genutzt): Der Gründer von Titus Skates hat das jährliche Titus Skates Open in Los Angeles organisiert, welches Street-Skate-Contests und urban-künstlerische Ausstellungen für lokale und internationale Talent kombiniert. Das Event umfasste Workshops zur Skateboard-Design-Entwicklung und förderte die Zusammenarbeit zwischen Skateboardern und Street-Art-Künstlern seit 2018.

Gefundener Kontext: ['Titus Skates wurde in den späten 1970er Jahren von dem deutschen Skateboard-Pionier Titus Dittmann gegründet.', 'Der Münsteraner Unternehmer Titus Dittmann organisierte den legendären Münster Monster Mastership.']

Finale Antwort: Den legendären Münster Monster Mastership



In [6]:
print(f"--- ANALYSE DER GEDANKENGÄNGE ---\n")
# Inspektion der letzten Interaktionen
local_llm.inspect_history(n=5)

--- ANALYSE DER GEDANKENGÄNGE ---





[2025-11-29T18:45:30.432848]

System message:

Your input fields are:
1. `question` (str): Die Frage des Nutzers
Your output fields are:
1. `reasoning` (str): 
2. `hypothetical_answer` (str): Eine plausible, hypothetische Antwort
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## hypothetical_answer ## ]]
{hypothetical_answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Schreibe eine hypothetische Antwort auf die Frage. Sie muss nicht faktisch korrekt sein, 
        sondern Keywords und Struktur enthalten, die in einem relevanten Dokument vorkommen könnten.


User message:

[[ ## question ## ]]
Was hat der Gründer von Titus Skates organisiert?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## hypothetical_answer ## ]]`, and then endin